# Velo render test (Kaggle)
Before running: right-side panel -> **Session options**: Accelerator = **GPU T4/P100**, **Internet = On**. Add your `velo-kit` dataset (the zip, unzipped automatically) via **Add Input**.

In [ ]:
!nvidia-smi | head -15
!nproc

In [ ]:
!apt-get update -qq && apt-get install -y -qq libegl1 libgl1 libxkbcommon0 libsm6 libxi6 libxxf86vm1 libxfixes3 libxrender1 xvfb libvulkan1 > /dev/null
!pip install -q bpy
import bpy; print('Blender', bpy.app.version_string)

In [ ]:
import glob, os, shutil, zipfile
os.makedirs('/kaggle/working/kit', exist_ok=True)
z = glob.glob('/kaggle/input/**/velo_kit.zip', recursive=True)
if z: zipfile.ZipFile(z[0]).extractall('/kaggle/working/kit')
else:
    src = glob.glob('/kaggle/input/*/')[0]; shutil.copytree(src, '/kaggle/working/kit', dirs_exist_ok=True)
print(os.listdir('/kaggle/working/kit'))

In [ ]:
# Build the 6 test stills of Velo and time them. ~200 s means CPU only; well under 60 s means the GPU is used.
import subprocess, time, os
K = '/kaggle/working/kit'
subprocess.run(['python3', f'{K}/tools/assemble.py', 'herotest', f'{K}/examples/hero_test_section.py', '--dur', '7', '--out', f'{K}/herotest_v1.py'], check=True)
open(f'{K}/herotest_mode.txt', 'w').write('stills:r1')
t = time.time()
r = subprocess.run(f'cd {K} && xvfb-run -a python3 tools/run_local.py {K} herotest', shell=True, capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-800:])
print('SECONDS', round(time.time() - t))

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('/kaggle/working/kit/heroteststills/r1/*.jpg')): display(Image(f, width=240))